# Chapitre 3 - Exercices 3.11 à 3.15

Régression à plusieurs réponses (3.11), ridge par augmentation des données (3.12), et
propriétés des méthodes à directions dérivées : PCR (3.13) et PLS (3.14, 3.15).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
set_style()

import pandas as pd
from scipy.optimize import minimize
from esl import ch03_regression as cr

Xtr, ytr, Xte, yte, names = cr.load_prostate_standardized()

## Exercice 3.11 : Régression multivariée pondérée

**Énoncé.** Montrer que la solution du problème de régression linéaire à plusieurs réponses
(3.40) est donnée par (3.39). Que se passe-t-il si les matrices de covariance $\Sigma_i$ sont
différentes pour chaque observation ?

**Rappels.** On a $K$ réponses : $\mathbf{Y}$ est $N\times K$ (ligne $i$ : $y_i^T$),
$\mathbf{X}$ est $N\times(p+1)$, $\mathbf{B}$ est $(p+1)\times K$ et $f(x_i) = \mathbf{B}^Tx_i$.
Les erreurs $\varepsilon_i \in \mathbb{R}^K$ ont une covariance $\Sigma$ (définie positive),
et l'on minimise le critère pondéré (éq. (3.40))

$$
\mathrm{RSS}(\mathbf{B}; \Sigma) = \sum_{i=1}^N (y_i - \mathbf{B}^Tx_i)^T\Sigma^{-1}(y_i - \mathbf{B}^Tx_i).
$$

La solution annoncée est (éq. (3.39)) $\hat{\mathbf{B}} = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{Y}$,
c'est-à-dire $K$ régressions **séparées**, sans tenir compte de $\Sigma$.

**Écriture matricielle.** Les vecteurs $y_i - \mathbf{B}^Tx_i$ sont les lignes de
$\mathbf{E}(\mathbf{B}) = \mathbf{Y} - \mathbf{X}\mathbf{B}$, donc

$$
\mathrm{RSS}(\mathbf{B}; \Sigma) = \sum_i \mathbf{e}_i^T\Sigma^{-1}\mathbf{e}_i = \mathrm{tr}\bigl[\mathbf{E}\,\Sigma^{-1}\mathbf{E}^T\bigr]
$$

(la trace d'une matrice est la somme de ses éléments diagonaux, ici les
$\mathbf{e}_i^T\Sigma^{-1}\mathbf{e}_i$).

**Décomposition.** Soit $\hat{\mathbf{B}} = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{Y}$ et
$\hat{\mathbf{E}} = \mathbf{Y} - \mathbf{X}\hat{\mathbf{B}}$ ; il vérifie les équations normales
$\mathbf{X}^T\hat{\mathbf{E}} = 0$. Pour $\mathbf{B}$ quelconque, posons $\Delta = \hat{\mathbf{B}} - \mathbf{B}$ :
$\mathbf{E}(\mathbf{B}) = \hat{\mathbf{E}} + \mathbf{X}\Delta$. En développant,

$$
\mathrm{RSS}(\mathbf{B}; \Sigma) = \mathrm{tr}[\hat{\mathbf{E}}\Sigma^{-1}\hat{\mathbf{E}}^T]
 + 2\,\mathrm{tr}[\mathbf{X}\Delta\Sigma^{-1}\hat{\mathbf{E}}^T]
 + \mathrm{tr}[\mathbf{X}\Delta\Sigma^{-1}\Delta^T\mathbf{X}^T]
$$

(les deux termes croisés sont égaux car $\mathrm{tr}(\mathbf{M}) = \mathrm{tr}(\mathbf{M}^T)$ et
$\Sigma^{-1}$ est symétrique).

- Le terme croisé est nul : par invariance de la trace par permutation circulaire,
  $\mathrm{tr}[\mathbf{X}\Delta\Sigma^{-1}\hat{\mathbf{E}}^T] = \mathrm{tr}[\Delta\Sigma^{-1}(\mathbf{X}^T\hat{\mathbf{E}})^T] = 0$.
- Le dernier terme est $\sum_i \mathbf{w}_i^T\Sigma^{-1}\mathbf{w}_i \ge 0$, où les
  $\mathbf{w}_i^T$ sont les lignes de $\mathbf{X}\Delta$, car $\Sigma^{-1}$ est définie positive ;
  il est nul ssi $\mathbf{X}\Delta = 0$, i.e. $\Delta = 0$ ($\mathbf{X}$ de rang plein).

Donc $\mathrm{RSS}(\mathbf{B}; \Sigma) \ge \mathrm{RSS}(\hat{\mathbf{B}}; \Sigma)$ avec égalité
ssi $\mathbf{B} = \hat{\mathbf{B}}$ : la solution de (3.40) est (3.39), **quelle que soit** $\Sigma$. $\square$

> **Intuition.** Les équations normales rendent le résidu orthogonal aux colonnes de
> $\mathbf{X}$ **colonne de réponse par colonne de réponse** ; la pondération $\Sigma^{-1}$ ne
> mélange que les colonnes de réponse entre elles et ne peut pas détruire cette orthogonalité.

**Covariances différentes $\Sigma_i$.** Le terme croisé devient
$2\sum_i \hat{\mathbf{e}}_i^T\Sigma_i^{-1}\Delta^Tx_i$, qui n'a plus de raison d'être nul : les
régressions ne se découplent plus. On se ramène à des moindres carrés pondérés (généralisés)
en vectorisant : notons $b = \mathrm{vec}(\mathbf{B})$ (colonnes de $\mathbf{B}$ empilées, longueur
$(p+1)K$). Avec le produit de Kronecker $\otimes$, on a
$\mathbf{B}^Tx_i = \mathbf{A}_ib$ où $\mathbf{A}_i = \mathbf{I}_K\otimes x_i^T$ (matrice $K\times(p+1)K$
dont la ligne $k$ contient $x_i^T$ dans le $k$-ième bloc). Le critère
$\sum_i(y_i - \mathbf{A}_ib)^T\Sigma_i^{-1}(y_i - \mathbf{A}_ib)$ est quadratique en $b$ ; en
annulant son gradient :

$$
\hat b = \Bigl(\sum_i\mathbf{A}_i^T\Sigma_i^{-1}\mathbf{A}_i\Bigr)^{-1}\sum_i\mathbf{A}_i^T\Sigma_i^{-1}y_i .
$$

Les $K$ équations sont couplées : l'estimation de chaque colonne de $\mathbf{B}$ utilise
toutes les réponses. Quand $\Sigma_i = \Sigma$ pour tout $i$, on retrouve (3.39).

**Vérification numérique.** On minimise le critère pondéré par optimisation numérique et on
compare aux formules.

In [2]:
rng = np.random.RandomState(0)
N, p, K = 80, 2, 2
X = cr.add_intercept(rng.randn(N, p))
B_true = np.array([[1.0, -1.0], [2.0, 0.5], [0.0, 1.5]])
Sigma = np.array([[1.0, 0.8], [0.8, 1.0]])
Y = X @ B_true + rng.multivariate_normal(np.zeros(K), Sigma, N)


def weighted_rss(b, Sig_inv):
    E = Y - X @ b.reshape(K, p + 1).T                  # b = vec(B), colonnes empilées
    return np.einsum("ik,ikl,il->", E, Sig_inv, E)


B_ols = np.linalg.solve(X.T @ X, X.T @ Y)
Sinv_const = np.repeat(np.linalg.inv(Sigma)[None], N, axis=0)
res = minimize(weighted_rss, np.zeros(K * (p + 1)), args=(Sinv_const,), method="BFGS", options={"gtol": 1e-10})
print("Sigma commun : écart entre optimum numérique et (3.39) = %.1e"
      % np.abs(res.x.reshape(K, p + 1).T - B_ols).max())

Sigma commun : écart entre optimum numérique et (3.39) = 1.5e-08


In [3]:
scales = rng.uniform(0.2, 3.0, N)
rhos = rng.uniform(-0.9, 0.9, N)
Sig_i = np.array([[[s**2, r * s], [r * s, 1.0]] for s, r in zip(scales, rhos)])   # définies positives
Sinv_i = np.linalg.inv(Sig_i)
A_i = [np.kron(np.eye(K), X[i][None, :]) for i in range(N)]
M = sum(A.T @ S @ A for A, S in zip(A_i, Sinv_i))
v = sum(A.T @ S @ Y[i] for i, (A, S) in enumerate(zip(A_i, Sinv_i)))
b_gls = np.linalg.solve(M, v)
res = minimize(weighted_rss, np.zeros(K * (p + 1)), args=(Sinv_i,), method="BFGS", options={"gtol": 1e-10})
print("Sigma_i variables : écart optimum numérique / formule vectorisée = %.1e" % np.abs(res.x - b_gls).max())
print("                    écart avec (3.39)                         = %.3f"
      % np.abs(b_gls.reshape(K, p + 1).T - B_ols).max())

Sigma_i variables : écart optimum numérique / formule vectorisée = 5.3e-08
                    écart avec (3.39)                         = 0.203


> **À retenir.** Avec une covariance $\Sigma$ commune, pondérer par $\Sigma^{-1}$ ne change
> rien : on fait $K$ régressions séparées. Si $\Sigma_i$ varie, il faut des moindres carrés
> généralisés qui couplent les réponses.

## Exercice 3.12 : Ridge par augmentation des données

**Énoncé.** Montrer que l'estimateur ridge s'obtient par moindres carrés ordinaires sur un
jeu de données **augmenté** : on ajoute à la matrice centrée $\mathbf{X}$ les $p$ lignes
$\sqrt\lambda\,\mathbf{I}_p$, et à $\mathbf{y}$ autant de zéros. Ces données artificielles de
réponse nulle forcent l'ajustement à rétrécir les coefficients vers 0 (idée des « indices »,
*hints*, d'Abu-Mostafa, 1995 : imposer une contrainte en ajoutant des exemples qui la
satisfont).

**Solution.** Posons

$$
\mathbf{X}_a = \begin{pmatrix}\mathbf{X}\\ \sqrt\lambda\,\mathbf{I}_p\end{pmatrix},\qquad
\mathbf{y}_a = \begin{pmatrix}\mathbf{y}\\ \mathbf{0}_p\end{pmatrix}.
$$

La norme d'un vecteur découpé en blocs est la somme des normes des blocs, donc

$$
\lVert\mathbf{y}_a - \mathbf{X}_a\beta\rVert^2 = \lVert\mathbf{y} - \mathbf{X}\beta\rVert^2 + \lVert\mathbf{0} - \sqrt\lambda\beta\rVert^2
 = \lVert\mathbf{y} - \mathbf{X}\beta\rVert^2 + \lambda\lVert\beta\rVert^2 ,
$$

qui est exactement le critère ridge (3.43). Les deux problèmes ont donc la même solution.
Matriciellement : $\mathbf{X}_a^T\mathbf{X}_a = \mathbf{X}^T\mathbf{X} + \lambda\mathbf{I}$ et
$\mathbf{X}_a^T\mathbf{y}_a = \mathbf{X}^T\mathbf{y}$, d'où
$(\mathbf{X}_a^T\mathbf{X}_a)^{-1}\mathbf{X}_a^T\mathbf{y}_a = \hat\beta^{\mathrm{ridge}}$. $\square$

Vérification sur la prostate (entrées centrées, $\lambda = 20$) :

In [4]:
lam = 20.0
Xc = Xtr - Xtr.mean(0)
yc = ytr - ytr.mean()
Xa = np.vstack([Xc, np.sqrt(lam) * np.eye(8)])
ya = np.r_[yc, np.zeros(8)]
beta_aug = cr.ols_qr(Xa, ya)
beta_ridge = cr.ridge_fit(Xtr, ytr, lam)[1]
pd.DataFrame({"MC augmentés": beta_aug, "ridge": beta_ridge}, index=names).round(4)

,MC augmentés,ridge
lcavol,0.4438,0.4438
lweight,0.2438,0.2438
age,-0.0573,-0.0573
lbph,0.1686,0.1686
svi,0.2349,0.2349
lcp,-0.0185,-0.0185
gleason,0.0383,0.0383
pgg45,0.1391,0.1391


> **Remarque.** Cette astuce permet de calculer ridge avec n'importe quel logiciel de
> moindres carrés (et de profiter de sa stabilité numérique, via la QR de $\mathbf{X}_a$). Les
> $p$ « pseudo-observations » $(\sqrt\lambda\,\mathbf{e}_j, 0)$ disent : « quand seule la
> variable $j$ vaut $\sqrt\lambda$, la réponse est nulle ».

> **À retenir.** Ridge = moindres carrés sur $(\mathbf{X}, \mathbf{y})$ complété par
> $(\sqrt\lambda\,\mathbf{I}_p, \mathbf{0})$.

## Exercice 3.13 : Coefficients de la PCR

**Énoncé.** Établir l'expression (3.62) et montrer que $\hat\beta^{\mathrm{pcr}}(p) = \hat\beta^{\mathrm{ls}}$.

**Rappels.** $\mathbf{X}$ est centrée (standardisée), de SVD $\mathbf{X} = \mathbf{U}\mathbf{D}\mathbf{V}^T$
avec $\mathbf{D}$ inversible. Les composantes principales sont $\mathbf{z}_m = \mathbf{X}\mathbf{v}_m = d_m\mathbf{u}_m$,
deux à deux orthogonales, et (éq. (3.61))
$\hat{\mathbf{y}}^{\mathrm{pcr}}_{(M)} = \bar y\mathbf{1} + \sum_{m=1}^M\hat\theta_m\mathbf{z}_m$
avec $\hat\theta_m = \langle\mathbf{z}_m, \mathbf{y}\rangle/\langle\mathbf{z}_m, \mathbf{z}_m\rangle$.

**Expression (3.62).** Par linéarité du produit matrice-vecteur,

$$
\sum_{m=1}^M\hat\theta_m\mathbf{z}_m = \sum_{m=1}^M\hat\theta_m\mathbf{X}\mathbf{v}_m = \mathbf{X}\Bigl(\sum_{m=1}^M\hat\theta_m\mathbf{v}_m\Bigr).
$$

L'ajustement PCR est donc l'ajustement linéaire $\bar y\mathbf{1} + \mathbf{X}\hat\beta$ de
coefficients

$$
\hat\beta^{\mathrm{pcr}}(M) = \sum_{m=1}^M\hat\theta_m\mathbf{v}_m .
$$

(Ces coefficients sont uniques car $\mathbf{X}$ est de rang plein.)

**Cas $M = p$.** Calculons $\hat\theta_m$ : $\langle\mathbf{z}_m, \mathbf{y}\rangle = d_m\mathbf{u}_m^T\mathbf{y}$
et $\langle\mathbf{z}_m, \mathbf{z}_m\rangle = d_m^2$ (car $\lVert\mathbf{u}_m\rVert = 1$), donc
$\hat\theta_m = \mathbf{u}_m^T\mathbf{y}/d_m$. D'où

$$
\hat\beta^{\mathrm{pcr}}(p) = \sum_{m=1}^p\mathbf{v}_m\frac{\mathbf{u}_m^T\mathbf{y}}{d_m} = \mathbf{V}\mathbf{D}^{-1}\mathbf{U}^T\mathbf{y}.
$$

Or $\mathbf{X}^T\mathbf{X} = \mathbf{V}\mathbf{D}^2\mathbf{V}^T$, donc
$(\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T = \mathbf{V}\mathbf{D}^{-2}\mathbf{V}^T\mathbf{V}\mathbf{D}\mathbf{U}^T = \mathbf{V}\mathbf{D}^{-1}\mathbf{U}^T$
($\mathbf{V}$ orthogonale). Ainsi $\hat\beta^{\mathrm{pcr}}(p) = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y} = \hat\beta^{\mathrm{ls}}$. $\square$

**Vérification numérique** : construction explicite $\sum_m\hat\theta_m\mathbf{v}_m$ pour tous
les $M$, et comparaison avec les moindres carrés pour $M = p$.

In [5]:
U, d, Vt = np.linalg.svd(Xc, full_matrices=False)
Z = Xc @ Vt.T
theta = Z.T @ yc / np.sum(Z**2, axis=0)
print("theta_m = u_m^T y / d_m ?", np.allclose(theta, U.T @ yc / d))
B_explicit = np.cumsum(theta[:, None] * Vt, axis=0)          # ligne M-1 : beta_pcr(M)
print("PCR(M) = sum theta_m v_m (module) ?", np.allclose(B_explicit, cr.pcr_path(Xtr, ytr)[1][1:]))
print("beta_pcr(p) =", B_explicit[-1].round(4))
print("beta_ls     =", cr.ols_fit(Xtr, ytr)["coef"][1:].round(4))

theta_m = u_m^T y / d_m ? True
PCR(M) = sum theta_m v_m (module) ? True
beta_pcr(p) = [ 0.6795  0.2631 -0.1415  0.2101  0.3052 -0.2885 -0.0213  0.267 ]
beta_ls     = [ 0.6795  0.2631 -0.1415  0.2101  0.3052 -0.2885 -0.0213  0.267 ]


> **À retenir.** $\hat\beta^{\mathrm{pcr}}(M) = \sum_{m\le M}\hat\theta_m\mathbf{v}_m$ avec
> $\hat\theta_m = \mathbf{u}_m^T\mathbf{y}/d_m$ ; pour $M = p$, $\mathbf{V}\mathbf{D}^{-1}\mathbf{U}^T\mathbf{y}$
> est la pseudo-inverse appliquée à $\mathbf{y}$, i.e. les moindres carrés.

## Exercice 3.14 : PLS avec entrées orthogonales

**Énoncé.** Montrer que, dans le cas orthogonal, PLS s'arrête après $m = 1$ étape, car les
$\hat\varphi_{mj}$ suivants (étape 2 de l'algorithme 3.3) sont nuls.